In [1]:
import gdspy

In [ ]:
# The GDSII file is called a library, which contains multiple cells.
lib = gdspy.GdsLibrary()

# Geometry must be placed in cells.
cell = lib.new_cell('FIRST')

# Create the geometry (a single rectangle) and add it to the cell.
rect = gdspy.Rectangle((0, 0), (2, 1))
cell.add(rect)

# Save the library in a file called 'first.gds'.
lib.write_gds(r'gds_output/first.gds')

# Optionally, save an image of the cell as SVG.
cell.write_svg(r'gds_output/first.svg')

# Display all cells using the internal viewer.
gdspy.LayoutViewer()


C:\Users\Quantum Circuits IB\AppData\Local\Temp\ipykernel_16884\3914110088.py:18: DeprecationWarning: [GDSPY] Use of the global library is deprecated.  Pass LayoutViewer a GdsLibrary instance.
  gdspy.LayoutViewer()


In [2]:
# The GDSII file is called a library, which contains multiple cells.
lib = gdspy.GdsLibrary()

# Geometry must be placed in cells.
cell = lib.new_cell('holes')


# Circle centered at (0, 0), with radius 2 and tolerance 0.1
circle = gdspy.Round((0, 0), 2, tolerance=0.01)
cell.add(circle)

# Save the library in a file called 'first.gds'.
lib.write_gds('gds_output/holes.gds')


### Array de Holes

### holes 3x10

In [2]:
layer_holes = {"layer": 4, "datatype": 0}

ysize = 3000 ## um
xsize = 10000 ## um
y0, yf = 580, 1100
# Espaciado entre círculos
spacing = 2 ##um
# Tamaño del array
rows = int((ysize -yf)/ spacing)
columns = int(xsize / spacing)

# Radio del círculo
radius = spacing / 4 ## um

# Tolerancia
tolerance = 0.01

# Crear la biblioteca
lib = gdspy.GdsLibrary()

# Crear el cell
#cell = lib.new_cell('holes_31')
cell = lib.new_cell('holes')

# Generar el array de círculos
for i in range(rows-25):
    for j in range(columns):
        # Calcular la posición del círculo actual
        x = 10 + j * spacing
        y = y0 + i * spacing
        
        # Crear el círculo y añadirlo al cell
        circle = gdspy.Round((x, y), radius, tolerance=tolerance, **layer_holes)
        cell.add(circle)

# Guardar la biblioteca en un archivo llamado 'holes.gds'
lib.write_gds(r'gds_output/holes_3_10_v3.gds')


### holes 10x10

In [8]:
layer_holes = {"layer": 1, "datatype": 3}

# Tamaño del array
rows = 200
columns = 200

# Espaciado entre círculos
spacing = 50.0 ##um

# Radio del círculo
radius = 10.0 ## um

# Tolerancia
tolerance = 0.01

# Crear la biblioteca
lib = gdspy.GdsLibrary()

# Crear el cell
cell = lib.new_cell('holes_10')

# Generar el array de círculos
for i in range(rows):
    for j in range(columns):
        # Calcular la posición del círculo actual
        x = 20 + j * spacing
        y = 20 + i * spacing
        
        # Crear el círculo y añadirlo al cell
        circle = gdspy.Round((x, y), radius, tolerance=tolerance, **layer_holes)
        cell.add(circle)

# Guardar la biblioteca en un archivo llamado 'holes.gds'
lib.write_gds('holes_10_10.gds')


### Array Holes

In [9]:
import klayout.db as db

In [27]:
## Initializing a new layout with a op-level cell.
ly = db.Layout()
top_cell = ly.create_cell("holes_single")

## Defining the horizontal and vertical spacing between the merged cells 
## and the initial x, y coordinates.
spacing_x = 100.0  # Separación horizontal entre GDS [um]
spacing_y = -100.0  # Separación vertical entre GDS [um]
x = 0  # Coordenada x inicial
y = 0  # Coordenada y inicial

four_ghz = ["holes_3_10.gds"]*4
five_ghz = ["holes_3_10.gds"]*4
six_ghz = ["holes_3_10.gds"]*4
for row in range(4):  # 3 filas
    for col in range(3):  # 3 columnas
        if col == 0:
            file = four_ghz[row]
        elif col == 1:
            file = five_ghz[row]
        else:
            file = six_ghz[row]

        ### creating a new layout instance ly_import, and the current GDS file is read into it. 
        ly_import = db.Layout()
        ly_import.read(file)
        ### The top cell ('Single_resonator') of the imported layout is stored in imported_top_cell.
        imported_top_cell = ly_import.top_cell()

        ## A new cell with the same name as the imported top cell is created in the main layout (ly). 
        target_cell = ly.create_cell(imported_top_cell.name)
        ## The contents of the imported top cell are then copied into the new cell.
        target_cell.copy_tree(imported_top_cell)

        # frees the resources of the imported layout
        ly_import._destroy()
        ## A new instance of the target cell is created and inserted into the top-level cell (top_cell)
        ## at the specified position.
        inst = db.DCellInstArray(target_cell.cell_index(), db.DTrans(db.DVector(x, y)))  # Cambio en la posición de la celda
        top_cell.insert(inst)
        ## Increment the x-coordinate by the width of the merged cell and the specified horizontal spacing.
        x += target_cell.dbbox().width() + spacing_x +30 # Incremento en la coordenada x
    
    ## Reset the x-coordinate and increment the y-coordinate by the height of the merged cell and the specified vertical spacing.
    x = 0  # Reiniciar la coordenada x
    y -= target_cell.dbbox().height() - spacing_y +30  # Incremento en la coordenada y

ly.write("array_single_holes.gds")


### holes 10x10

In [29]:
ly = db.Layout()
top_cell = ly.create_cell("holes_16_32px")

## Defining the horizontal spacing between the merged cells 

## and the initial x, y coordinates.
spacing_x = 100.0  # Separación horizontal entre GDS [um]
spacing_y = -100.0  # Separación vertical entre GDS [um]
x = 0  # Coordenada x inicial
y = 0  # Coordenada y inicial

files = ['holes_10_10.gds']*5
files2 = ['holes_10_10.gds']*5
for row in range(2):  # 2 filas (0,1)
    for col in range(5):  # 5 columnas (0,1,2,3,4)
        file = files[col] if row % 2 == 0 else files2[col]  # Alternar entre los archivos especificados

        ### creating a new layout instance ly_import, and the current GDS file is read into it. 
        ly_import = db.Layout()
        ly_import.read(file)
        ### The top cell ('Single_resonator') of the imported layout is stored in imported_top_cell.
        imported_top_cell = ly_import.top_cell()

        ## A new cell with the same name as the imported top cell is created in the main layout (ly). 
        target_cell = ly.create_cell(imported_top_cell.name)
        ## The contents of the imported top cell are then copied into the new cell.
        target_cell.copy_tree(imported_top_cell)

        # frees the resources of the imported layout
        ly_import._destroy()
        ## A new instance of the target cell is created and inserted into the top-level cell (top_cell)
        ## at the specified position.
        inst = db.DCellInstArray(target_cell.cell_index(), db.DTrans(db.DVector(x, y)))  # Cambio en la posición de la celda
        top_cell.insert(inst)
        ## Increment the x-coordinate by the width of the merged cell and the specified horizontal spacing.
        x += target_cell.dbbox().width() + spacing_x +30  # Incremento en la coordenada x
    
    ## Reset the x-coordinate and increment the y-coordinate by the height of the merged cell and the specified vertical spacing.
    x = 0  # Reiniciar la coordenada x
    y -= target_cell.dbbox().height() - spacing_y +30 # Incremento en la coordenada y

ly.write("array_16_32px_holes.gds")


#### Joinning holes

In [30]:
ly = db.Layout()
top_cell = ly.create_cell("TOP_I_holes")

## Defining the horizontal spacing between the merged cells 

## and the initial x, y coordinates.
spacing_x = 1.0  # Separación horizontal entre GDS [um]
spacing_y = 1.0  # Separación vertical entre GDS [um]
x = 0  # Coordenada x inicial
y = 0  # Coordenada y inicial

files= [ "array_single_holes.gds"]
files1= [ "array_16_32px_holes.gds"]

for row in range(2):  # 3 filas (0,1)
    for col in range(1):  # 2 columnas (0,1,2,3,4)
        file = files[col] if row % 2 == 0 else files1[col]  # Alternar entre los archivos especificados

        ### creating a new layout instance ly_import, and the current GDS file is read into it. 
        ly_import = db.Layout()
        ly_import.read(file)
        ### The top cell ('Single_resonator') of the imported layout is stored in imported_top_cell.
        imported_top_cell = ly_import.top_cell()

        ## A new cell with the same name as the imported top cell is created in the main layout (ly). 
        target_cell = ly.create_cell(imported_top_cell.name)
        ## The contents of the imported top cell are then copied into the new cell.
        target_cell.copy_tree(imported_top_cell)

        # frees the resources of the imported layout
        ly_import._destroy()
        ## A new instance of the target cell is created and inserted into the top-level cell (top_cell)
        ## at the specified position.
        inst = db.DCellInstArray(target_cell.cell_index(), db.DTrans(db.DVector(x, y)))  # Cambio en la posición de la celda
        top_cell.insert(inst)
        ## Increment the x-coordinate by the width of the merged cell and the specified horizontal spacing.
        x += target_cell.dbbox().width() + spacing_x  # Incremento en la coordenada x
        y -= target_cell.dbbox().height() -spacing_y  # Incremento en la coordenada y
    ## Reset the x-coordinate and increment the y-coordinate by the height of the merged cell and the specified vertical spacing.
    x = -10.1e3  # Reiniciar la coordenada x
    y=-19.4e3
    
    print(target_cell.dbbox().height())
ly.write("holes_part_one.gds")


12269.984
20069.992


In [31]:
ly = db.Layout()
top_cell = ly.create_cell("TOP_II_holes")

## Defining the horizontal spacing between the merged cells 

## and the initial x, y coordinates.
spacing_x = 1.0  # Separación horizontal entre GDS [um]
spacing_y = 1.0  # Separación vertical entre GDS [um]
x = 0  # Coordenada x inicial
y = 0  # Coordenada y inicial

files= [ "holes_part_one.gds"]
files1= [ "array_single_holes.gds"]

for row in range(2):  # 3 filas (0,1)
    for col in range(1):  # 2 columnas (0,1,2,3,4)
        file = files[col] if row % 2 == 0 else files1[col]  # Alternar entre los archivos especificados

        ### creating a new layout instance ly_import, and the current GDS file is read into it. 
        ly_import = db.Layout()
        ly_import.read(file)
        ### The top cell ('Single_resonator') of the imported layout is stored in imported_top_cell.
        imported_top_cell = ly_import.top_cell()

        ## A new cell with the same name as the imported top cell is created in the main layout (ly). 
        target_cell = ly.create_cell(imported_top_cell.name)
        ## The contents of the imported top cell are then copied into the new cell.
        target_cell.copy_tree(imported_top_cell)

        # frees the resources of the imported layout
        ly_import._destroy()
        ## A new instance of the target cell is created and inserted into the top-level cell (top_cell)
        ## at the specified position.
        inst = db.DCellInstArray(target_cell.cell_index(), db.DTrans(db.DVector(x, y)))  # Cambio en la posición de la celda
        top_cell.insert(inst)
        ## Increment the x-coordinate by the width of the merged cell and the specified horizontal spacing.
        x += target_cell.dbbox().width() + spacing_x  # Incremento en la coordenada x
        y -= target_cell.dbbox().height() -spacing_y  # Incremento en la coordenada y
    ## Reset the x-coordinate and increment the y-coordinate by the height of the merged cell and the specified vertical spacing.
    x = 0e3  # Reiniciar la coordenada x
    y=-32.6e3
    
    print(target_cell.dbbox().height())
ly.write("holes_array_final.gds")


32469.992000000002
12269.984


## Porbando unir un chip con los holes

In [22]:
## Initializing a new layout with a op-level cell.
ly = db.Layout()
top_cell = ly.create_cell("join_Resonators")
## Defining the vertical spacing between the merged cells 
## and the initial y-coordinate.

spacing = 1.0 ## separacion vertical entre gds [um]
y = 0 ## initial point - vertical
x = 0 ## initial point - horizontal
files= [ "prueba.gds", "holes_3_10.gds"]
files2= [ "holes_3_10.gds"]

for file in files: 
  ### creating a new layout instance ly_import, and the current GDS file is read into it. 
  ly_import = db.Layout()
  ly_import.read(file)
  ### The top cell ('Single_resonator') of the imported layout is stored in imported_top_cell.
  imported_top_cell = ly_import.top_cell()
  
  ## A new cell with the same name as the imported top cell is created in the main layout (ly). 
  target_cell = ly.create_cell(imported_top_cell.name)
  ## The contents of the imported top cell are then copied into the new cell.
  target_cell.copy_tree(imported_top_cell)
  
  # frees the resources of the imported layout
  ly_import._destroy()
  ## A new instance of the target cell is created and inserted into the top-level cell (top_cell)
  ## at the specified position.
  inst = db.DCellInstArray(target_cell.cell_index(), db.DTrans(db.DVector(0, y)))
  top_cell.insert(inst)
  ## The y-coordinate is incremented by the height of the merged cell and the specified spacing.
  y = 0#target_cell.dbbox().height()

ly.write("join_a.gds")

### Uniendo el array final

In [33]:
## Initializing a new layout with a op-level cell.
ly = db.Layout()
top_cell = ly.create_cell("mkids_mask")
## Defining the vertical spacing between the merged cells 
## and the initial y-coordinate.

spacing = 1.0 ## separacion vertical entre gds [um]
y = 0 ## initial point - vertical
x = 0 ## initial point - horizontal
files= [ "mask_design_final_v2.gds", "holes_array_final.gds"]

for file in files: 
  ### creating a new layout instance ly_import, and the current GDS file is read into it. 
  ly_import = db.Layout()
  ly_import.read(file)
  ### The top cell ('Single_resonator') of the imported layout is stored in imported_top_cell.
  imported_top_cell = ly_import.top_cell()
  
  ## A new cell with the same name as the imported top cell is created in the main layout (ly). 
  target_cell = ly.create_cell(imported_top_cell.name)
  ## The contents of the imported top cell are then copied into the new cell.
  target_cell.copy_tree(imported_top_cell)
  
  # frees the resources of the imported layout
  ly_import._destroy()
  ## A new instance of the target cell is created and inserted into the top-level cell (top_cell)
  ## at the specified position.
  inst = db.DCellInstArray(target_cell.cell_index(), db.DTrans(db.DVector(0, y)))
  top_cell.insert(inst)
  ## The y-coordinate is incremented by the height of the merged cell and the specified spacing.
  y = 0#target_cell.dbbox().height()

ly.write("mask_mkids_final.gds")